# Rail Pulse — 3. Human evaluation and model comparison
Label **50 collected posts yourself before viewing model predictions**. Compare the primary multilingual CardiffNLP transformer against a second pretrained English social-media model. Record three or more concrete failure explanations. This notebook does not invent labels or accuracy.

In [ ]:
%pip install -q pandas transformers torch scikit-learn

In [ ]:
from pathlib import Path
import sqlite3, os
DB = Path('rail_pulse.sqlite3').resolve()
MODEL = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
BASELINE = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
def connect():
    db=sqlite3.connect(str(DB),timeout=30,check_same_thread=False)
    db.execute('PRAGMA journal_mode=WAL')
    db.execute('PRAGMA busy_timeout=30000')
    db.execute('CREATE TABLE IF NOT EXISTS posts (id TEXT PRIMARY KEY, created_at TEXT NOT NULL, collected_at TEXT NOT NULL, text TEXT NOT NULL, category TEXT NOT NULL, disruption INTEGER NOT NULL, sentiment TEXT NOT NULL, confidence REAL NOT NULL, model TEXT NOT NULL)')
    db.execute('CREATE TABLE IF NOT EXISTS heartbeat (id INTEGER PRIMARY KEY CHECK(id=1), last_seen TEXT, events_seen INTEGER, reconnects INTEGER)')
    db.execute('CREATE TABLE IF NOT EXISTS sessions (id INTEGER PRIMARY KEY AUTOINCREMENT, started_at TEXT, last_seen TEXT, stopped_at TEXT)')
    db.commit();return db

In [ ]:
import pandas as pd
from pathlib import Path
LABEL_FILE=Path('human_labels_50.csv')
with connect() as db:
    count=db.execute('SELECT COUNT(*) FROM posts').fetchone()[0]
    print('Collected posts:',count)
    if count>=50 and not LABEL_FILE.exists():
        rows=pd.read_sql_query('SELECT id,text,category,disruption FROM posts ORDER BY RANDOM() LIMIT 50',db)
        rows['human_label']=''
        rows.to_csv(LABEL_FILE,index=False,encoding='utf-8-sig')
        print('Created',LABEL_FILE,'— manually fill human_label before proceeding')
    elif count<50:print('Wait until 50 real items have been collected')
    else:print('Existing labels file preserved:',LABEL_FILE)

## Human labelling
Open `human_labels_50.csv` and label each row `positive`, `neutral`, or `negative`. **Do not inspect model predictions before completing your labels.** Decide whether you're evaluating all transport posts or only disruption posts and state that choice in your report.

In [ ]:
from transformers import pipeline
from sklearn.metrics import classification_report,confusion_matrix
if not LABEL_FILE.exists():
    print('No labels file yet')
else:
    labels=pd.read_csv(LABEL_FILE,encoding='utf-8-sig',keep_default_na=False)
    labels['human_label']=labels['human_label'].str.strip().str.lower()
    valid={'positive','neutral','negative'}
    if len(labels)<50 or not labels.human_label.isin(valid).all():
        print('Complete all 50 human labels before running inference')
    else:
        for model_name,column in [(MODEL,'model1_label'),(BASELINE,'model2_label')]:
            clf=pipeline('text-classification',model=model_name,tokenizer=model_name,device=-1)
            predictions=clf(labels.text.tolist(),truncation=True,max_length=512,batch_size=8)
            labels[column]=[{'label_0':'negative','label_1':'neutral','label_2':'positive'}.get(p['label'].lower(),p['label'].lower()) for p in predictions]
            print('MODEL:',model_name)
            print(classification_report(labels.human_label,labels[column],labels=['negative','neutral','positive'],zero_division=0))
        labels['primary_correct']=labels.human_label==labels.model1_label
        labels['secondary_correct']=labels.human_label==labels.model2_label
        labels['failure_explanation']=''
        labels.to_csv('model_comparison_50.csv',index=False,encoding='utf-8-sig')
        print('Saved model_comparison_50.csv; fill failure_explanation for at least three primary-model errors.')
        display(labels[~labels.primary_correct][['text','human_label','model1_label','model2_label','failure_explanation']].head(15))

## Report checklist
- Report accuracy for **both** models on the **same 50 human-labelled items**.
- Explain at least three failures: negation, sarcasm, Dutch expressions, mixed sentiment, announcements versus reactions, etc.
- Compare model cards: training data, languages, labels, licences and limitations.
- Ethical reflection: public posts exclude many passengers; automated accounts can dominate; store no handles; show counts beside sentiment; require human review before decisions.
- Verify the running collector spans 48 hours and the SQLite database has at least 300 real, relevant items. No backfilled or synthetic posts count toward live collection.